# 02 — Limpeza e Feature Engineering

Objetivos:
- Padronizar colunas e valores
- Tratar datas e ruídos
- Construir visão **cliente-atual** (1 linha por cliente, registro mais recente por data_ordem)
- Criar features recomendadas (nunca_acessou, log_acessos, faixa_inatividade, parcelado etc.)
- Salvar bases processadas em `data/processed/`

In [ ]:
from pathlib import Path
import sys

sys.path.append(str(Path('..').resolve()))


import pandas as pd

from src.features import build_modeling_dataframe
from src.preprocessing import (
    build_cliente_atual,
    clean_base,
    load_xlsx_base_clientes,
    winsorize_df,
)

PROJECT_ROOT = Path('..').resolve()
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'Base Customer Sucess.xlsx'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df_raw = load_xlsx_base_clientes(str(RAW_PATH))
df_trans, meta = clean_base(df_raw)
df_trans.shape, meta

## 1) Tratamento de outliers (decisão aplicada)

- `n_acessos` já tem `log_acessos = log1p(n_acessos)`
- `dias_sem_acessar` é winsorizado em 1%–99% para reduzir influência de extremos no K-Means

In [ ]:
df_trans = winsorize_df(df_trans, cols=[c for c in ['dias_sem_acessar'] if c in df_trans.columns], lower_q=0.01, upper_q=0.99)
df_trans[['dias_sem_acessar']].describe().T if 'dias_sem_acessar' in df_trans.columns else None

## 2) Visão cliente-atual (1 linha por cliente)

Regra: manter o registro mais recente por `data_ordem` e **mesclar agregações históricas** para capturar múltiplas compras sem duplicar clientes.

In [ ]:
df_cliente = build_cliente_atual(df_trans)
df_cliente.shape, df_cliente['cliente'].nunique(), df_cliente.head(3)

## 3) Dataset final de modelagem

Selecionamos features comportamentais e financeiras (com versões numéricas e flags) e evitamos identificadores/alta cardinalidade.

In [ ]:
X, spec = build_modeling_dataframe(df_cliente)
spec, X.shape, X.head(3)

## 4) Salvando artefatos (processed)
- `base_limpa_transacao.parquet`
- `base_cliente_atual.parquet`
- `modeling_X.parquet`

In [ ]:
df_trans.to_parquet(PROCESSED_DIR / 'base_limpa_transacao.parquet', index=False)
df_cliente.to_parquet(PROCESSED_DIR / 'base_cliente_atual.parquet', index=False)
X.to_parquet(PROCESSED_DIR / 'modeling_X.parquet', index=False)

pd.DataFrame({'tipo': ['numeric']*len(spec.numeric) + ['categorical']*len(spec.categorical), 'feature': spec.numeric + spec.categorical}).to_json(PROCESSED_DIR / 'feature_spec.json', orient='records', force_ascii=False)
str(PROCESSED_DIR)